# Pavement 2024: GeoPackage to GeoJSON
GEOG 464 Lab 4 Recipe 3 workflow: read, select columns, simplify in metres, reproject, export, and check.

In [1]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import geopandas as gpd
import os

# Support running from the project folder or the notebooks folder.
data_dir = 'data' if os.path.isdir('data') else '../data'
input_file = os.path.join(data_dir, 'auscultation-chaussee-2024.gpkg')
output_file = os.path.join(data_dir, 'pavement_2024.geojson')

gdf = gpd.read_file(input_file, layer='Auscultation chaussée 2024')
print('CRS:', gdf.crs)
print('Number of features:', len(gdf))
print('Geometry types:', gdf.geom_type.unique())
print('Column names:', gdf.columns.tolist())

CRS: EPSG:32188
Number of features: 17176
Geometry types: <StringArray>
['LineString']
Length: 1, dtype: str
Column names: ['ID_TRC', 'Rue', 'De', 'A', 'Longueur', 'Arrondissement', 'DateReleve', 'Indice_PCI', 'Etat_PCI', 'Indice_IRI', 'Etat_IRI', 'geometry']


In [3]:
gdf = gdf[['Rue', 'Arrondissement', 'Indice_PCI', 'Etat_PCI', 'geometry']].copy()

# The source CRS (EPSG:32188) uses metres. Simplify before reprojecting.
gdf["geometry"] = gdf.geometry.simplify(5)
gdf = gdf.to_crs(epsg=4326)

gdf.to_file(output_file, driver='GeoJSON', COORDINATE_PRECISION=6)
written = gpd.read_file(output_file)
file_size_mb = os.path.getsize(output_file) / 1_000_000
print('Number of features written:', len(written))
print(f'Output file size: {file_size_mb:.2f} MB')

Number of features written: 17176
Output file size: 4.66 MB


In [4]:
# Check the exported GeoJSON.
check = gpd.read_file(output_file)
problems = []

if len(check) != 17176:
    problems.append(f'Expected 17176 features; found {len(check)}.')
if check.crs is None or check.crs.to_epsg() != 4326:
    problems.append('CRS must be EPSG:4326.')

# Montreal area: west, south, east, north (longitude/latitude).
west, south, east, north = -74.1, 45.3, -73.4, 45.8
bounds = check.geometry.bounds
inside = (
    (bounds.minx >= west) & (bounds.maxx <= east)
    & (bounds.miny >= south) & (bounds.maxy <= north)
)
if not inside.all() or check.geometry.isna().any() or check.geometry.is_empty.any():
    problems.append('Some geometries are missing, empty, or outside the Montreal bounding box.')

popup_fields = ['Rue', 'Arrondissement', 'Indice_PCI', 'Etat_PCI']
missing_fields = [field for field in popup_fields if field not in check.columns]
if missing_fields:
    problems.append(f'Missing popup fields: {missing_fields}')

file_size_mb = os.path.getsize(output_file) / 1_000_000
if file_size_mb > 5:
    problems.append(f'File size exceeds 5 MB: {file_size_mb:.2f} MB.')

print('Geometry types:', check.geom_type.unique())
if problems:
    for problem in problems:
        print('CHECK FAILED:', problem)
else:
    print('All checks passed')

Geometry types: <StringArray>
['LineString']
Length: 1, dtype: str
All checks passed


In [5]:
# Diagnostic only: check the ORIGINAL data for possible duplicate road segments

raw_check = gpd.read_file(
    "../data/auscultation-chaussee-2024.gpkg",
    layer="Auscultation chaussée 2024"
)

geom_key = raw_check.geometry.to_wkb()
check = raw_check.assign(_geom_key=geom_key)

print("Rows:", len(raw_check))

if "ID_TRC" in raw_check.columns:
    print("Duplicate ID_TRC values:", raw_check["ID_TRC"].duplicated().sum())
else:
    print("ID_TRC is not present in the source data.")

print(
    "Rows involved in identical geometries:",
    check["_geom_key"].duplicated(keep=False).sum()
)

conflicting_geometries = (
    check.groupby("_geom_key")["Etat_PCI"]
    .nunique()
    .gt(1)
    .sum()
)

print(
    "Identical geometries with more than one pavement condition:",
    conflicting_geometries
)

Rows: 17176
Duplicate ID_TRC values: 0
Rows involved in identical geometries: 0
Identical geometries with more than one pavement condition: 0
